In [11]:
!pip -q install -U "protobuf<6.30"
!pip -q install -U transformers sentencepiece sacrebleu pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 120.6 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires r

In [12]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

dtype = torch.float16 if device == "cuda" else torch.float32

# Tương thích các transformers phiên bản khác nhau
try:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        low_cpu_mem_usage=True,
    ).to(device)
except TypeError:
    # Fallback cho vài phiên bản cũ hơn
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    ).to(device)

model.eval()


Device: cuda
GPU: Tesla T4


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
!pip install gdown
!gdown "https://drive.google.com/drive/folders/1ofo-8p1v-mTFbeV2tPspymDPMhKDYCid?usp=sharing" --folder

In [ ]:
import pandas as pd

DATA_PATH = "Alignment/JSON2_4653_5815.csv"
df = pd.read_csv(DATA_PATH)

# Lấy 50 dòng đầu
df50 = df.head(50).dropna(subset=["src_lang", "tgt_lang"]).copy()

sources = df50["src_lang"].astype(str).tolist()   # tiếng Trung
refs = df50["tgt_lang"].astype(str).tolist()      # tiếng Việt (reference)

print("Columns:", list(df.columns))
print("Using rows:", len(df50))
print("Example source:", sources[0])
print("Example ref:", refs[0])


Columns: ['src_id', 'src_lang', 'tgt_lang']
Using rows: 50
Example source: 我认为这将是主要的事情。
Example ref: Tôi nghĩ rằng đó sẽ là điều chính.


In [ ]:
import math
from tqdm import tqdm
import sacrebleu

# Chinese -> Vietnamese
SRC_LANG = "zh_CN"
TGT_LANG = "vi_VN"

BATCH_SIZE = 8
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4

OUT_PATH = "kaggle/working/mbart50_preds_first50.csv"

tokenizer.src_lang = SRC_LANG
forced_bos_token_id = tokenizer.lang_code_to_id.get(TGT_LANG)

@torch.inference_mode()
def generate_batch(texts):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SOURCE_LEN,
    ).to(device)

    if device == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model.generate(
                **enc,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=NUM_BEAMS,
                max_new_tokens=MAX_NEW_TOKENS,
                early_stopping=True,
            )
    else:
        out = model.generate(
            **enc,
            forced_bos_token_id=forced_bos_token_id,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            early_stopping=True,
        )

    return tokenizer.batch_decode(out, skip_special_tokens=True)

In [16]:
preds = []
steps = math.ceil(len(sources) / BATCH_SIZE)

for i in tqdm(range(steps), desc="Generating"):
    batch = sources[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    preds.extend(generate_batch(batch))

# Metrics
bleu = sacrebleu.corpus_bleu(preds, [refs])
chrf = sacrebleu.corpus_chrf(preds, [refs])
ter = sacrebleu.corpus_ter(preds, [refs])

print(f"SacreBLEU: {bleu.score:.2f}")
print(f"chrF:      {chrf.score:.2f}")
print(f"TER:       {ter.score:.2f}")

# Save predictions
out_df = df50.copy()
out_df["prediction"] = preds
out_df.to_csv("mbart50_preds_first50.csv", index=False, encoding="utf-8-sig")
print("\nSaved: mbart50_preds_first50.csv")

out_df[["src_id", "src_lang", "tgt_lang", "prediction"]].head(5)


Generating: 100%|██████████| 7/7 [00:07<00:00,  1.05s/it]


SacreBLEU: 21.22
chrF:      38.79
TER:       67.60

Saved: mbart50_preds_first50.csv


,src_id,src_lang,tgt_lang,prediction
0,4653,我认为这将是主要的事情。,Tôi nghĩ rằng đó sẽ là điều chính.,Tôi nghĩ đó sẽ là điều quan trọng.
1,4653,我认为，同意某人所说，应该选择HSV-1血清阳性且携带APOE4等位基因的人。,"Tôi nghĩ, đồng ý với người nào đó nói rằng nên...","Tôi cho rằng, tôi đồng ý với một người đã nói ..."
2,4654,我非常想说这个。,Tôi rất muốn nói điều này.,Tôi rất muốn nói điều này.
3,4654,史蒂文·雅各布森：但是马克，我可以提个建议吗？,"Steven Jacobson: Nhưng Mack, tôi có thể đưa ra...","Steven Jacobson: Nhưng Mark, tôi có một lời đề..."
4,4654,我相信这是从许多问题中得出的结论。,Tôi chắc chắn điều này được tìm kiếm từ rất nh...,Tôi tin rằng đó là kết quả từ rất nhiều vấn đề.
